In [6]:
from pathlib import Path
import pandas as pd

PKL_PATH = Path("/mnt/Volume/Mega/PHD/Bocconi/code/hd-epic-annotations/narrations-and-action-segments/HD_EPIC_Narrations.pkl")

if not PKL_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {PKL_PATH}")

df = pd.read_pickle(PKL_PATH)

print(f"Loaded pickle file: {PKL_PATH}")
print(f"DataFrame shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

print("\nColumns and dtypes:")
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))

print("\nPreview:")
display(df.head())

Loaded pickle file: /mnt/Volume/Mega/PHD/Bocconi/code/hd-epic-annotations/narrations-and-action-segments/HD_EPIC_Narrations.pkl
DataFrame shape: 59,454 rows x 16 columns

Columns and dtypes:


,column,dtype
0,unique_narration_id,object
1,participant_id,object
2,video_id,object
3,narration,object
4,start_timestamp,float64
5,end_timestamp,float64
6,nouns,object
7,verbs,object
8,pairs,object
9,main_actions,object



Preview:


,unique_narration_id,participant_id,video_id,narration,start_timestamp,end_timestamp,nouns,verbs,pairs,main_actions,verb_classes,noun_classes,pair_classes,main_action_classes,hands,narration_timestamp
0,P01-20240202-110250-1,P01,P01-20240202-110250,Open the upper cupboard by holding the handle...,7.44,8.75,"[upper cupboard, handle of cupboard]","[open, hold]","[(open, upper cupboard), (hold, handle of cupb...","[(open, upper cupboard)]","[3, 34]","[3, 3]","[(3, 3), (34, 3)]","[(3, 3)]",[left hand],8.000000
1,P01-20240202-110250-2,P01,P01-20240202-110250,Stretch the left hand inside the cupboard in ...,8.85,9.36,"[cupboard, lower shelf, mug, far side of cupbo...","[stretch, reach]","[(stretch, left hand), (reach, mug)]","[(stretch, left hand)]","[50, 98]","[3, 247, 13, 3]","[(50, 11), (98, 13)]","[(50, 11)]",[left hand],9.666667
2,P01-20240202-110250-3,P01,P01-20240202-110250,Pick up a mug from the lower shelf of the cup...,9.34,11.16,"[mug, lower shelf of cupboard]",[pick up],"[(pick up, mug)]","[(pick up, mug)]",[0],"[13, 3]","[(0, 13)]","[(0, 13)]",[left hand],10.833333
3,P01-20240202-110250-4,P01,P01-20240202-110250,Turn the mug in the left hand to face the cam...,11.15,11.60,"[mug, camera]","[turn, face]","[(turn, mug), (face, camera)]","[(turn, mug)]","[23, 11]","[13, 297]","[(23, 13), (11, 297)]","[(23, 13)]",[left hand],11.466667
4,P01-20240202-110250-5,P01,P01-20240202-110250,Close the cupboard using the right hand.,11.50,12.18,[cupboard],[close],"[(close, cupboard)]","[(close, cupboard)]",[4],[3],"[(4, 3)]","[(4, 3)]",[right hand],11.566667


In [7]:
import json

CSV_PATH = PKL_PATH.with_name("unofficial_narrations_converted_from_pkl.csv")

def normalize_nested(value):
    if isinstance(value, set):
        return sorted(normalize_nested(v) for v in value)
    if isinstance(value, tuple):
        return [normalize_nested(v) for v in value]
    if isinstance(value, list):
        return [normalize_nested(v) for v in value]
    if isinstance(value, dict):
        return {str(k): normalize_nested(v) for k, v in sorted(value.items(), key=lambda item: str(item[0]))}
    return value

def serialize_nested(value):
    if isinstance(value, (list, dict, tuple, set)):
        return json.dumps(normalize_nested(value), ensure_ascii=False, sort_keys=True)
    return value

object_columns = df.select_dtypes(include="object").columns
df_csv = df.copy()

for col in object_columns:
    df_csv[col] = df_csv[col].map(serialize_nested)

df_csv.to_csv(CSV_PATH, index=False)

print(f"Saved CSV to: {CSV_PATH}")
print(f"CSV shape: {df_csv.shape[0]:,} rows x {df_csv.shape[1]} columns")
print(f"Serialized object columns: {len(object_columns)}")

Saved CSV to: /mnt/Volume/Mega/PHD/Bocconi/code/hd-epic-annotations/narrations-and-action-segments/unofficial_narrations_converted_from_pkl.csv
CSV shape: 59,454 rows x 16 columns
Serialized object columns: 13


In [10]:
import numpy as np

csv_check = pd.read_csv(CSV_PATH)

df_expected = df.copy()
for col in object_columns:
    df_expected[col] = df_expected[col].map(serialize_nested)

same_shape = df_expected.shape == csv_check.shape
same_columns = df_expected.columns.tolist() == csv_check.columns.tolist()

if not (same_shape and same_columns):
    raise AssertionError("Validation failed: CSV shape or column order differs from the source DataFrame.")

numeric_columns = [
    col for col in df_expected.columns
    if pd.api.types.is_numeric_dtype(df_expected[col])
 and col not in object_columns
]
text_columns = [col for col in df_expected.columns if col not in numeric_columns]

numeric_mismatches = {}
for col in numeric_columns:
    left_num = pd.to_numeric(df_expected[col], errors="coerce").to_numpy(dtype=float)
    right_num = pd.to_numeric(csv_check[col], errors="coerce").to_numpy(dtype=float)
    equal_mask = np.isclose(left_num, right_num, rtol=0.0, atol=1e-9, equal_nan=True)
    numeric_mismatches[col] = int((~equal_mask).sum())

text_mismatches = {}
for col in text_columns:
    left_text = df_expected[col].fillna("__NaN__").astype(str)
    right_text = csv_check[col].fillna("__NaN__").astype(str)
    text_mismatches[col] = int((left_text != right_text).sum())

total_numeric_mismatches = int(sum(numeric_mismatches.values()))
total_text_mismatches = int(sum(text_mismatches.values()))
total_mismatches = total_numeric_mismatches + total_text_mismatches

validation = pd.Series(
    {
        "same_shape": same_shape,
        "same_columns": same_columns,
        "numeric_columns_checked": len(numeric_columns),
        "text_columns_checked": len(text_columns),
        "total_numeric_mismatches": total_numeric_mismatches,
        "total_text_mismatches": total_text_mismatches,
        "total_mismatches": total_mismatches,
    },
    name="result",
)

print(f"Read CSV from: {CSV_PATH}")
display(validation.to_frame())

if total_mismatches > 0:
    raise AssertionError(
        "Validation failed: exported CSV differs from the source DataFrame. "
        f"Mismatched cells: {total_mismatches}"
    )

print("Validation passed: CSV export is consistent with the source DataFrame.")

Read CSV from: /mnt/Volume/Mega/PHD/Bocconi/code/hd-epic-annotations/narrations-and-action-segments/unofficial_narrations_converted_from_pkl.csv


,result
same_shape,True
same_columns,True
numeric_columns_checked,3
text_columns_checked,13
total_numeric_mismatches,0
total_text_mismatches,0
total_mismatches,0


Validation passed: CSV export is consistent with the source DataFrame.
